# Optional Lab: Understanding Redis and Building a Job Queue

**From a key-value store to an extension of our Docker Compose lab**

This notebook is an introduction, a commented live-demo script, and a student assignment.
All examples run without OpenAI or Stability keys: a slow Python function simulates the
work performed by image generation in our existing lab.

**Learning objectives:** Explain what Redis stores and how clients access it; experiment
with strings, hashes, TTLs, and lists; distinguish producers, queues, and workers; run an
RQ job without a web server and then through FastAPI; finally integrate Redis and a worker
into your existing Compose lab.

**Suggested timing:** Redis basics and direct demos: about 25 minutes; queues/RQ: about
25 minutes; FastAPI: about 20 minutes; independent Compose extension: about 45–60 minutes.
You should already know basic Python, HTTP/FastAPI, and the preceding Docker lab.

> **For the instructor:** Each demo includes a prediction question and an expected
> observation. After setup, run the code cells from top to bottom. Workers and the demo
> API run in separate processes; the final code cell shuts them down.


## 1 · What Is Redis?

Redis is a **separate server for data structures**. A client, such as our Python program,
sends commands over a network connection. The server manages keys and values and returns
results. `redis-py` is the Python client library, **not the server**.

The active dataset is held primarily in memory (RAM), which makes many operations very
fast. Redis stores more than simple text values: it supports hashes, lists, sets, and
sorted sets, among other structures. Common uses include caches, session data, counters,
leaderboards, and storing tasks for workers.

![Redis clients, keys and data structures in RAM, with optional persistence](figures/redis-basics.svg)

| Term | Meaning | Example |
|---|---|---|
| Key | A unique name within a Redis database | `lab:visits` |
| String | A sequence of bytes; numbers can also be stored this way | `SET lab:name Ada` |
| Hash | Multiple fields under one key | A person's name and course |
| List | An ordered sequence with insertion/removal at both ends | A queue |
| TTL | A key's remaining lifetime in seconds | A cache entry expires after 30 seconds |

Data travels over the network as bytes. In our first examples, `decode_responses=True`
tells the client to turn those bytes into text. For RQ, we deliberately use a **second
client without text decoding**, because RQ also stores binary job data.


### How Do Commands, Expiration, and Persistence Work?

1. The client sends a command such as `INCR lab:visits`.
2. Redis applies the command to its data structure and returns the new value.
3. Several clients can access the same key. `INCR` is atomic: concurrent increments are
   not lost through competing reads and writes. A sequence you implement yourself,
   `GET → +1 → SET`, is **not automatically atomic**.

Many commands are inexpensive, but Redis does not have unlimited performance. Large
values, expensive operations, network latency, and available RAM all matter. Saying
“Redis is single-threaded” is too simplistic as an overall description. For our examples,
what matters is that individual commands are processed atomically, not that every internal
server operation uses only one thread.

**In memory does not necessarily mean temporary:** RDB writes snapshots, while AOF logs
changes. How much data can be lost during a failure depends on the configuration. With
containers, the persistence settings **and** a volume for the data directory must work
together. A volume alone does not enable AOF. A TTL is independent of this: a key can
expire even when persistence is enabled.

**Think about it:** What is the difference between “the cache entry expired” and “the
Redis container was deleted”?


## 2 · Live-Demo Setup

Use **Python 3.12**, Docker, and Linux/macOS or **WSL2 on Windows**. Native Windows workers
are not supported by this RQ version because it relies on Unix signals. Use a Jupyter-capable
editor, such as VS Code with the Python and Jupyter extensions. The demo uses RQ 1.15.1,
matching the existing backend; current RQ tutorials may show APIs from newer versions.

In a terminal, starting from the repository directory:

```bash
cd optional-lab-redis
python3.12 -m venv .venv
source .venv/bin/activate       # On Windows, run these commands in WSL2.
python -m pip install -r requirements.txt
python -m ipykernel install --user --name optional-redis --display-name "Python (Redis Lab)"
```

Open this notebook and select the **Python (Redis Lab)** kernel. If you use JupyterLab
instead of VS Code, also run `pip install jupyterlab`, followed by `jupyter lab`.

Start a dedicated Redis container in a **second terminal**:

```bash
docker run --rm --name optional-redis -p 127.0.0.1:16379:6379 redis:7.4-alpine
```

This terminal remains occupied. Redis listens on port 6379 inside the container; the
notebook kernel on the host reaches it through **127.0.0.1:16379**. Binding to 127.0.0.1
keeps the demo local. It uses neither a production Redis database nor cloud API keys.
If the container is already running, do not start another one with the same name.

If the kernel runs **inside a container**, `localhost` refers to that container. Set
`REDIS_URL` to a host or Redis service name reachable from there. For now, this notebook
assumes that the kernel and the published Docker port are on the same host.

**No Compose solution yet:** A single Redis container is enough for these demos.
Connecting the services with Compose is your assignment later.


In [ ]:
import os
import sys
import time
import json
import uuid
from pathlib import Path

# Start the notebook from the repository root or from optional-lab-redis.
LAB_DIR = Path.cwd()
if (LAB_DIR / "optional-lab-redis" / "redis-introduction.ipynb").exists():
    LAB_DIR = LAB_DIR / "optional-lab-redis"
assert (LAB_DIR / "redis-introduction.ipynb").exists(), "Start from the repository or notebook directory."
os.chdir(LAB_DIR)

# Do not load the image backend's .env files: this demo does not need API keys.
REDIS_URL = os.getenv("REDIS_URL", "redis://127.0.0.1:16379/0")
SESSION = uuid.uuid4().hex[:8]  # Separate keys and queue for each notebook run.
PREFIX = f"optional-lab:{SESSION}"
QUEUE_NAME = f"optional-lab-{SESSION}"
RUNTIME = LAB_DIR / ".runtime"
RUNTIME.mkdir(exist_ok=True)
print("Python:", sys.version.split()[0], "| Demo prefix:", PREFIX)


In [ ]:
import redis

# Text client for our own strings, hashes, and lists.
redis_text = redis.Redis.from_url(
    REDIS_URL, decode_responses=True, socket_connect_timeout=3, socket_timeout=5
)
assert redis_text.ping() is True  # PING checks whether the server is reachable.
print("Redis responds: PONG")


**Expected:** `Redis responds: PONG`.
If you get `ConnectionError`, check whether the container is running, the port mapping
is correct, and the kernel is actually on the same host. Use `docker ps` and
`docker logs optional-redis` to investigate. If you get `ModuleNotFoundError`, check the
selected notebook kernel.


## 3 · Redis Without FastAPI: Values, Counters, Hashes, and TTL

**Predict:** What Python type does `GET` return for a counter? Why do you not need to
use `GET` followed by `SET` to implement a safe shared counter?


In [ ]:
name_key = f"{PREFIX}:name"
counter_key = f"{PREFIX}:visits"
redis_text.set(name_key, "Ada")
redis_text.set(counter_key, 0)

# INCR is a single atomic Redis command.
for _ in range(3):
    visits = redis_text.incr(counter_key)

print("Name:", redis_text.get(name_key))
print("INCR result:", visits, type(visits).__name__)
print("GET result:", redis_text.get(counter_key), type(redis_text.get(counter_key)).__name__)
assert visits == 3


**Observe:** `INCR` returns the number `3`, while `GET` returns the string `"3"` when
using this text client. Redis commands have different response types.


In [ ]:
profile_key = f"{PREFIX}:student"
redis_text.hset(profile_key, mapping={"name": "Ada", "course": "Solution Deployment"})
redis_text.hset(profile_key, "completed_labs", 2)
redis_text.hincrby(profile_key, "completed_labs", 1)
print(redis_text.hgetall(profile_key))  # A hash holds multiple fields under one key.


In [ ]:
ttl_key = f"{PREFIX}:temporary"
redis_text.set(ttl_key, "temporarily available", ex=2)  # ex sets a TTL in seconds.
print("Value immediately:", redis_text.get(ttl_key))
print("TTL immediately:", redis_text.ttl(ttl_key))
time.sleep(2.2)  # Deliberately wait for expiration during the live demo.
print("Value afterward:", redis_text.get(ttl_key))
print("TTL afterward:", redis_text.ttl(ttl_key))
assert redis_text.get(ttl_key) is None


**Expected:** The temporary value is gone and `TTL` returns `-2`. A return value of
`-1` means the key exists but has no expiration time.

**Mini-exercise (3 minutes):** Create your own session key with a 10-second TTL. Read it
immediately and after it expires. Explain why a TTL is useful for a cache but needs to be
chosen particularly carefully for jobs that have not yet been processed. Continue using
`PREFIX` for your own keys.


## 4 · What Is a Queue, and Why Do We Want One?

A queue is a **waiting line for work**. A producer adds tasks; a consumer or worker
retrieves and processes them. With FIFO (*first in, first out*), the first task enqueued
is the first one removed.

Image generation in our image lab can take several seconds. If it happens directly
within an HTTP request, the client waits for the finished image. A queue provides:

- **Fast acceptance:** The API responds with a job ID before the image is ready.
- **Decoupling:** API and worker run in separate processes; workers can start later.
- **Buffering:** A temporary burst of requests can wait when workers are at capacity.
- **Independent scaling:** Additional workers can process several jobs in parallel.

In return, we need another service, status queries, and a strategy for failures, retries,
and overload. A queue **does not make an individual job faster**. A queue that keeps
growing means more work is arriving than the workers can process.

![Producer, Redis queue, workers and results; FIFO does not describe completion order](figures/queue-flow.svg)

**Keep the roles separate:** Redis stores data about tasks. A Python worker executes
the Python function. Redis does not run our image generator itself. Different workers
may finish jobs at different speeds: FIFO removal does not guarantee FIFO completion.


### Comparing Three Approaches

| Approach | When does the client get the result? | Where does the work run? | Limitation |
|---|---|---|---|
| Direct function call / HTTP request | After the work finishes | Calling process / API process | Long request; timeouts are possible |
| FastAPI `BackgroundTasks` | Acceptance first; retrieve the result later | Still within the API application | No external persistent job queue |
| Redis + RQ | Job ID first; retrieve the result later | Separate worker | Additional infrastructure and error handling |

`async`/`await` does not replace a worker queue. It helps with cooperative waiting for
I/O, but does not automatically move CPU work into another process. For the synchronous
libraries in our demo, we use regular `def` routes; FastAPI runs them in its thread pool.


## 5 · The Smallest Queue: A Redis List, Without RQ or FastAPI

**Predict:** In what order will A, B, and C be processed?
`RPUSH` appends on the right; `BLPOP` removes from the left. Combining `RPUSH` and `BRPOP`
would **not produce FIFO behavior** here.


In [ ]:
list_key = f"{PREFIX}:simple-queue"
redis_text.delete(list_key)  # Reset only our own demo list.
for label in ["A", "B", "C"]:
    # JSON keeps this simple demo's payload readable.
    redis_text.rpush(list_key, json.dumps({"task": label}))
print("Waiting tasks:", redis_text.llen(list_key))
print("List contents:", redis_text.lrange(list_key, 0, -1))


In [ ]:
processed = []
for _ in range(3):  # Deliberately finite: Run All must not block forever.
    item = redis_text.blpop(list_key, timeout=2)
    if item is None:
        break
    queue_key, payload = item
    task = json.loads(payload)
    time.sleep(0.2)  # Stand-in for expensive processing.
    processed.append(task["task"])
    print("Processed:", task["task"])
assert processed == ["A", "B", "C"]
print("Empty queue:", redis_text.blpop(list_key, timeout=1))


**Observe:** A, B, C, followed by `None` after the timeout. `BLPOP` waits on the
consumer's connection; it does not block the entire Redis server. Other clients can
still execute commands.

**This mini-queue's limitation:** Once a job has been popped, it is no longer in the
list. If the consumer crashes afterward, there is no acknowledgment mechanism and the
job can be lost. We have not yet implemented job IDs, states, results, retries, or failure
tracking. An empty list also does not mean that all jobs already removed from it have finished.

More reliable list queues require additional mechanisms. Redis Streams, for example,
provide consumer groups and acknowledgments. For our Python lab, we will use **RQ (Redis
Queue)** instead of building our own solution.


## 6 · RQ Without FastAPI: A Real Python Task

RQ adds Python jobs, status tracking, results, workers, and failure tracking to Redis.
Among other things, a job contains the function's import path and its arguments. The
producer and worker must both be able to import the same function.

**Notebook pitfall:** A function defined directly in a notebook usually belongs to
`__main__`. RQ cannot reliably import it in the worker. The next cell therefore writes a
small module into `.runtime/`. These generated demo files are not committed to Git.
You can run the cell again; it overwrites only this demo module.

**Predict:** After `enqueue()`, will any Python task code run if no worker is running yet?


In [ ]:
%%writefile .runtime/redis_lab_tasks.py
import os
import time


def build_report(name: str, delay: float = 1.0) -> dict:
    # Simulate slow work: no API key or external charges required.
    time.sleep(delay)
    if name == "FAIL":
        # An intentional failure for the later live demo.
        raise ValueError("Intentional demo error")
    return {
        "name": name,
        "word_count": len(name.split()),
        "worker_pid": os.getpid(),  # Shows which process performed the work.
    }


In [ ]:
import importlib
from rq import Queue
from rq.job import Job

# Both the kernel and the worker process must be able to find the module.
if str(RUNTIME) not in sys.path:
    sys.path.insert(0, str(RUNTIME))
importlib.invalidate_caches()
import redis_lab_tasks
importlib.reload(redis_lab_tasks)

# RQ needs bytes, not automatically decoded strings!
redis_jobs = redis.Redis.from_url(REDIS_URL, decode_responses=False)
queue = Queue(QUEUE_NAME, connection=redis_jobs)
tracked_jobs = []

started = time.perf_counter()
job = queue.enqueue(
    redis_lab_tasks.build_report, "My first report", 1.0,
    job_timeout=30, result_ttl=300, failure_ttl=300,
)
tracked_jobs.append(job)
print("Enqueue took:", round(time.perf_counter() - started, 3), "seconds")
print("Status without a worker:", job.get_status())
print("Waiting jobs:", queue.count)
assert job.get_status() == "queued"


**Expected:** `queued`. Redis knows about the job, but nobody is working on it yet.
`job_timeout` limits execution time; `result_ttl` controls retention after success, and
`failure_ttl` controls retention after failure. These values are not the job's waiting
time in the queue.

RQ uses Pickle serialization by default. Producers and Redis must therefore be trusted;
untrusted job data must not be allowed into this queue unchecked.

### Start a Separate Worker

For a short notebook demo on Linux/macOS or WSL2, we use **`rq.SimpleWorker`** in a
separate process. It executes jobs directly in that process and does not have the same
process isolation or heartbeat behavior as a regular RQ worker. The existing Docker lab
still targets `python worker.py` with the regular worker. `SpawnWorker`, shown in newer
RQ documentation, is not available in our version, 1.15.1.

The helper functions below start only this notebook's processes, write logs to `.runtime/`,
and support targeted cleanup. We do not start processes with `&` or use a global `pkill`.


In [ ]:
import atexit
import subprocess

processes = {}  # name -> (process, log file)


def stop_process(name):
    record = processes.pop(name, None)
    if record is None:
        return
    process, log = record
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=8)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
    log.close()


def cleanup_processes():
    for name in list(processes):
        stop_process(name)


atexit.register(cleanup_processes)  # In addition to the explicit cleanup cell at the end.


def start_process(name, arguments):
    assert name not in processes, f"{name} is already running; call stop_process first."
    env = os.environ.copy()
    env.update(REDIS_URL=REDIS_URL, RQ_QUEUE_NAME=QUEUE_NAME)
    log = (RUNTIME / f"{SESSION}-{name}.log").open("w", encoding="utf-8")
    process = subprocess.Popen(arguments, cwd=RUNTIME, env=env, stdout=log, stderr=log)
    processes[name] = (process, log)
    return process


def start_worker(name):
    # Run the installed RQ CLI using the same Python as the notebook kernel.
    return start_process(name, [
        sys.executable, "-c", "from rq.cli import main; main()", "worker", "--url", REDIS_URL,
        "--worker-class", "rq.SimpleWorker", "--name", f"{PREFIX}-{name}", QUEUE_NAME,
    ])


def wait_for_job(job, timeout=20):
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        state = job.get_status(refresh=True)
        if state in {"finished", "failed", "stopped", "canceled"}:
            return state
        time.sleep(0.1)
    raise TimeoutError("Job did not finish. Check the worker log in .runtime/.")


In [ ]:
worker_process = start_worker("worker-1")
assert wait_for_job(job) == "finished"
print("Result:", job.return_value())
print("Notebook-PID:", os.getpid(), "| Worker-PID:", worker_process.pid)
assert job.return_value()["worker_pid"] != os.getpid()


**Observe:** The first job is processed after the worker starts. The PID shows that
the code ran in a different process. We did not need an API for this. RQ stores this small
Python result in Redis; our image lab, by contrast, saves JPEG files to the filesystem.

**Live experiment:** Stop the worker after a job has finished. Enqueue new work, then
start a worker again. Does the queued job remain available while Redis stays running?


In [ ]:
stop_process("worker-1")  # No job is running now; we are not interrupting a task.
paused_job = queue.enqueue(redis_lab_tasks.build_report, "Process later", 0.5, result_ttl=300)
tracked_jobs.append(paused_job)
time.sleep(0.3)
print("Worker stopped; status:", paused_job.get_status())
assert paused_job.get_status() == "queued"

start_worker("worker-1")
assert wait_for_job(paused_job) == "finished"
print("After restart:", paused_job.return_value())


### Make Failures Visible

**Predict:** Can `enqueue()` succeed even though the function later raises an error?
Accepting a job is not the same thing as completing it.


In [ ]:
failed_job = queue.enqueue(redis_lab_tasks.build_report, "FAIL", 0.2, failure_ttl=300)
tracked_jobs.append(failed_job)
assert wait_for_job(failed_job) == "failed"
failed_job.refresh()  # Load failure details from Redis.
print("Status:", failed_job.get_status())
print("Error:", failed_job.exc_info.strip().splitlines()[-1])


RQ records the failure; it **does not automatically retry** this job here. Retries
would need to be configured explicitly. RQ also does not guarantee that every business
action happens exactly once. Tasks that may be retried should be **idempotent** where
possible, so they do not, for example, charge the same payment twice.

**Mini-exercise (5 minutes):** Start a second worker with `start_worker("worker-2")`,
enqueue three reports, and compare their `worker_pid` values. Use the same `result_ttl`
and `failure_ttl` values as above for each job and collect the jobs in `tracked_jobs`.
Then stop `worker-2`. Explain why a short job started later might finish before a long
one. More parallelism does not automatically reduce each individual report's runtime.


## 7 · The Same Principle With FastAPI

FastAPI now becomes the producer. We provide two approaches:

- `POST /reports/sync`: Compute the report during the request and return the result.
- `POST /reports`: Enqueue the job and return **202 Accepted** with a job ID.
- `GET /reports/{job_id}`: Check status; once `finished`, the result appears in `result`.

HTTP 202 means **accepted**, not “finished.” A job may fail later. This demo returns
HTTP 200 for known job-status queries, with the job's state in the JSON response; an
unknown or expired job ID returns 404. The existing image lab has a different contract:
`POST /v3/images` already returns HTTP 200 with `img_id`, while polling
`GET /images/{img_id}` returns 202 or a JPEG with HTTP 200.

**Predict:** Which POST approach responds sooner? Does the queue itself make the report
compute faster?


In [ ]:
%%writefile .runtime/redis_lab_api.py
import os

import redis
from fastapi import FastAPI, HTTPException
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field
from rq import Queue
from rq.job import Job
from rq.exceptions import NoSuchJobError

from redis_lab_tasks import build_report

app = FastAPI(title="Redis Queue – Live Demo")
connection = redis.Redis.from_url(os.environ["REDIS_URL"], decode_responses=False)
queue = Queue(os.environ["RQ_QUEUE_NAME"], connection=connection)


class ReportRequest(BaseModel):
    name: str = Field(min_length=1, max_length=80)
    delay: float = Field(default=1.0, ge=0, le=5)


@app.exception_handler(redis.exceptions.RedisError)
async def redis_unavailable(request, exc):
    # An infrastructure failure must not look like successful job acceptance.
    return JSONResponse(status_code=503, content={"detail": "Redis is unavailable."})


@app.post("/reports/sync")
def report_sync(request: ReportRequest):
    # Regular def: the blocking demo function runs in FastAPI's thread pool.
    return build_report(request.name, request.delay)


@app.post("/reports", status_code=202)
def enqueue_report(request: ReportRequest):
    # The API does NOT execute build_report here. The worker will do that later.
    job = queue.enqueue(
        build_report, request.name, request.delay,
        job_timeout=30, result_ttl=300, failure_ttl=300,
    )
    return {"job_id": job.id, "status_url": f"/reports/{job.id}"}


@app.get("/reports/{job_id}")
def report_status(job_id: str):
    try:
        job = Job.fetch(job_id, connection=connection)
    except NoSuchJobError:
        raise HTTPException(status_code=404, detail="Job is unknown or has already expired.")
    if job.origin != queue.name:
        raise HTTPException(status_code=404, detail="Job does not belong to this demo.")
    state = job.get_status(refresh=True)
    body = {"job_id": job.id, "status": state}
    if state == "finished":
        body["result"] = job.return_value()  # Check the status, not just whether result is None!
    elif state == "failed":
        body["error"] = "Report failed; check the worker log."
    return body


In [ ]:
import socket
import httpx

# Choose a free local port to avoid interfering with existing applications.
with socket.socket() as sock:
    sock.bind(("127.0.0.1", 0))
    api_port = sock.getsockname()[1]
API_URL = f"http://127.0.0.1:{api_port}"
api_process = start_process("api", [
    sys.executable, "-m", "uvicorn", "redis_lab_api:app",
    "--host", "127.0.0.1", "--port", str(api_port),
])
http = httpx.Client(base_url=API_URL, timeout=10, trust_env=False)

# Wait for readiness; starting Popen does not mean the server is ready.
deadline = time.monotonic() + 15
while time.monotonic() < deadline:
    if api_process.poll() is not None:
        raise RuntimeError("API exited; check its log in .runtime/.")
    try:
        response = http.get("/openapi.json")
        if response.status_code == 200:
            break
    except httpx.ConnectError:
        pass
    time.sleep(0.1)
else:
    raise TimeoutError("API did not become ready.")
print("Interactive API documentation:", API_URL + "/docs")


In [ ]:
# Compare the same simulated work with two different response timings.
payload = {"name": "FastAPI Live Report", "delay": 1.5}
started = time.perf_counter()
sync_response = http.post("/reports/sync", json=payload)
sync_response.raise_for_status()
print("Direct response after", round(time.perf_counter() - started, 2), "seconds:")
print(sync_response.json())

started = time.perf_counter()
queued_response = http.post("/reports", json=payload)
assert queued_response.status_code == 202
print("Job accepted after", round(time.perf_counter() - started, 2), "seconds:")
print(queued_response.json())
api_job = Job.fetch(queued_response.json()["job_id"], connection=redis_jobs)
tracked_jobs.append(api_job)


In [ ]:
def poll_report(status_url, timeout=20):
    deadline = time.monotonic() + timeout
    previous = None
    while time.monotonic() < deadline:
        response = http.get(status_url)
        response.raise_for_status()
        body = response.json()
        if body["status"] != previous:
            print("Status:", body["status"])
            previous = body["status"]
        if body["status"] in {"finished", "failed", "stopped", "canceled"}:
            return body
        time.sleep(0.2)  # Poll with a pause and an overall deadline.
    raise TimeoutError("No terminal state: is a worker listening to the correct queue?")

report = poll_report(queued_response.json()["status_url"])
assert report["status"] == "finished"
print("Result:", report["result"])


**Observe:** The direct response takes at least the simulated processing time. The
queued response returns an ID quickly, but the result still only appears after processing
finishes. Depending on timing, polling may skip some intermediate states.

The API process ID in the synchronous result differs from the worker PID in the queued
result. The separate worker is responsible for processing the queue throughout the demo.

**Browser alternative for the live demo:** Open the printed `/docs` URL, submit a report
with `delay=3`, and then query its ID. In a remote development environment, you may need
to forward the printed port so the browser can reach it.


In [ ]:
# Observe a failure through the same HTTP interface.
response = http.post("/reports", json={"name": "FAIL", "delay": 0.2})
assert response.status_code == 202  # Acceptance succeeds even though the task fails later.
tracked_jobs.append(Job.fetch(response.json()["job_id"], connection=redis_jobs))
failed_report = poll_report(response.json()["status_url"])
assert failed_report["status"] == "failed"
print(failed_report)

assert http.get("/reports/not-found").status_code == 404
assert http.post("/reports", json={"name": "", "delay": 1}).status_code == 422
print("Unknown job: 404; invalid input: 422")


### Check Your Understanding Before Applying This to the Lab

1. Why can `POST /reports` succeed even though the job later fails?
2. What happens if Redis is reachable but no worker is running?
3. How do the API process, Redis server, and worker process differ?
4. Where are this demo's RQ results stored? Where are our lab's images stored instead?
5. When would you deliberately keep a direct request or use `BackgroundTasks`?
6. What additional questions arise with retries, many jobs, or a restart?

**Discussion point:** Results have a lifetime. An expired job is not the same as a failed
job. A real application would also need an access-control design, backpressure,
idempotency, and an appropriate persistence strategy.


## 8 · Clean Up the Demo

The next cell stops only our worker/API processes and removes our tracked jobs and demo
keys. **No `FLUSHDB` and no global shutdown of all containers.** If you want to repeat an
earlier demo, restart the notebook with a fresh kernel and run it from the beginning.
This gives you a new isolated queue and key prefix.


In [ ]:
cleanup_processes()  # Stop workers even if the API section has not run yet.
if "http" in globals():
    http.close()
for demo_job in tracked_jobs:
    # Jobs may already have been removed by their TTL.
    try:
        demo_job.delete()
    except Exception as exc:
        from rq.exceptions import NoSuchJobError
        if not isinstance(exc, NoSuchJobError):
            raise
queue.delete(delete_jobs=True)  # Only the randomly named queue belonging to this run.
redis_text.delete(name_key, counter_key, profile_key, ttl_key, list_key)
redis_jobs.close()
redis_text.close()
print("Demo processes stopped; our own jobs and keys removed.")


When you no longer need the demo container, stop that specific container in a terminal:

```bash
docker stop optional-redis
```

It was started with `--rm`, so it will be removed. This demo deliberately did not configure
a persistence volume. You can keep the generated Python files and logs in `.runtime/`
for your next presentation; Git ignores them.


## 9 · Your Assignment: Add Redis to the Existing Compose Lab

**Task:** Extend your solution from [lab-assignments.ipynb](../lab-assignments.ipynb) so
that image generation runs through Redis and a separate RQ worker. Work in
[`docker-lab/docker-compose.yml`](../docker-lab/docker-compose.yml). Completing the
Dockerfiles remains your task; this notebook does not provide a finished Compose file to copy.

![Target architecture: frontend, backend, Redis, worker, and a shared image volume](figures/compose-architecture.svg)

### Step A – Identify the Roles in the Existing Code

Read these files and identify the producer, queue connection, worker, and result storage:

- [`backend/routes/image_gen_router.py`](../docker-lab/backend/routes/image_gen_router.py):
  `POST /v3/images` enqueues `worker.gen_image_task`. The response contains `img_id` (HTTP 200).
- [`backend/dependencies.py`](../docker-lab/backend/dependencies.py):
  `REDIS_URL` controls the connection; when no queue name is specified, RQ uses `default`.
- [`backend/worker.py`](../docker-lab/backend/worker.py):
  `python worker.py` listens to `high`, `default`, and `low` and saves the generated JPEG.
- `GET /images/{img_id}` returns 202 while the file does not exist, then returns the JPEG.
- [`frontend/streamlit_app.py`](../docker-lab/frontend/streamlit_app.py):
  Select **v3** to actually use the Redis path.

**When applying the demo:** Use the existing endpoints and `img_id`, not the demo's
`/reports` routes. Image generation requires valid provider credentials and account
credits; the Redis/report demos above require neither.


### Step B – Plan and Configure Four Services

Add **`redis`** and **`worker`** services alongside `frontend` and `backend`. The worker
can use the same built image as the backend, but with a different startup command:
`python worker.py`. Use a suitable Python version, such as Python 3.12.

| Location | Configuration value / requirement | Your explanation |
|---|---|---|
| Frontend | `API_URL=http://backend:8000` | Why not `localhost`? |
| Backend **and** worker | `REDIS_URL=redis://redis:6379/0` | Why the same server and database? |
| Backend **and** worker | The same absolute `UPLOAD_FOLDER` | Which process writes and which reads? |
| Worker | Start with `python worker.py`; share the backend's task modules | Why is the Redis container alone insufficient? |
| Redis | Port 6379 within the Compose network | Does this port need to be published on the host? |

In Compose, the **service name is the DNS name**. Inside a container, `localhost` refers
to that container. Communication between containers uses the container port, not an
arbitrary host-port mapping. Redis does not need to be reachable from the host for this
application. Provide the required provider variables at runtime, for example through
`env_file` or `environment`; do not bake credentials into the image.

**Think first, then write YAML:** Which two processes can actually generate images?
Which settings are simply service URLs? Which settings must the backend and worker share?


### Step C – Readiness and Two Different Volumes

1. **Redis readiness:** Define a healthcheck using `redis-cli ping`. The backend and worker
   should wait for a healthy Redis service. `depends_on` without a health condition only
   sets startup order, not readiness. It also does not automatically handle connection
   failures that occur after startup.
2. **Image volume:** Mount the same named volume in the backend and worker at the same
   `UPLOAD_FOLDER`. Ensure that the directory exists and is writable. When running locally
   from `backend/`, you may need `mkdir -p images`.
3. **Redis persistence:** For this exercise, enable AOF (`--appendonly yes`) and mount a
   separate Redis data volume at `/data`. Explain why an image volume does not protect
   Redis data, and why a Redis volume does not contain the worker's JPEG files.
4. Use `docker compose down` for the restart experiment **without `-v`**. The `-v` option
   would delete the named volumes, changing the very persistence test you are trying to run.

**Submit:** Your updated Compose file and, if applicable, Dockerfiles; a small architecture
diagram; and short answers to the questions in the table. Include evidence for the tests
below, such as terminal output with job/image IDs and successful requests. Do not include
provider keys in screenshots or submission text.


### Step D – Acceptance Tests and Experiments

Run the commands from `docker-lab/`. If you chose different service names, adjust the
commands accordingly.

1. **Startup:** Run `docker compose up --build -d` and check the state with
   `docker compose ps`. Redis should be healthy, and the API and worker should be running.
2. **Happy path:** Submit a job through Streamlit v3 or `POST /v3/images`. Poll the returned
   `img_id` with `GET /images/{img_id}`. You may initially receive 202, followed by 200 with
   a JPEG. Watch `docker compose logs -f worker`.
3. **Pause the worker:** Run `docker compose stop worker`, then submit another job. It stays
   in the queue and the image is missing. Continue with `docker compose start worker`:
   the existing job should be processed without submitting it again.
4. **Persistence:** Stop the worker, create exactly one queued job, and note its ID.
   Shut down the stack with `docker compose down`. Start it again without rebuilding;
   AOF and the Redis volume should preserve the job. Also check whether a previously
   generated image remains available through the image volume.
5. **Scaling (optional):** Run `docker compose up -d --scale worker=2`. Submit several jobs
   and observe the worker logs. Do not assign a fixed `container_name` to a worker service
   that you want to scale.

**Success criteria:** A request enters the existing `default` queue; a separate worker
processes it; the backend serves the result from the shared volume; pausing a worker does
not lose waiting jobs; Redis persistence and image persistence are explained and tested
separately.

**Bonus / a real limitation of the existing lab:** The current image-polling endpoint
does not distinguish a running job from an unknown ID or a failed job. If a provider fails,
the frontend may therefore keep polling indefinitely. Propose how you would add a mapping
from `img_id` to RQ job ID, failure status, and a polling timeout. This is an extension,
not a prerequisite for the basic assignment.


## 10 · References and Further Reading

The following official documentation explains the concepts used here. The demo deliberately
uses the versions in `requirements.txt`; examples for newer RQ versions are not always
directly transferable.

- [Redis: Data types](https://redis.io/docs/latest/develop/data-types/)
- [Redis: INCR and atomicity](https://redis.io/docs/latest/commands/incr/)
- [Redis: TTL](https://redis.io/docs/latest/commands/ttl/)
- [Redis: BLPOP and the limitations of a basic list queue](https://redis.io/docs/latest/commands/blpop/)
- [Redis: Persistence with RDB and AOF](https://redis.io/docs/latest/operate/oss_and_stack/management/persistence/)
- [RQ: Queues, jobs, and function requirements](https://python-rq.org/docs/)
- [RQ: Connections and decoding](https://python-rq.org/docs/connections/)
- [RQ: Workers and SimpleWorker](https://python-rq.org/docs/workers/)
- [FastAPI: Background Tasks](https://fastapi.tiangolo.com/tutorial/background-tasks/)
- [Docker Compose: Networking](https://docs.docker.com/compose/how-tos/networking/)
- [Docker Compose: Startup order and readiness](https://docs.docker.com/compose/how-tos/startup-order/)
- [Docker Compose: Volumes](https://docs.docker.com/reference/compose-file/volumes/)

The SVG diagrams in `figures/` are part of this notebook and can also be opened directly
for a presentation.
